# SNCP-PPO Social Navigation - Colab Notebook

End-to-end notebook for training and evaluating an **LTC + PPO** crowd-aware navigation policy.

## Run order
1. **Setup** - clone repo, install deps, optional Drive mount
2. **Smoke test** - verify env + model + a tiny training loop
3. **Train** - vectorized training in the paper's scenario
4. **Evaluate** - density sweep in the paper scenario
5. **Visualize** - trajectory plot + GIFs
6. **Analyze** - learning curves from the training CSV
7. **Persist** - download checkpoint + evidence bundle

## Current run: v32 - v30 + curriculum N->25 + budget 4M, single-variable

v22/v23 fell short of the paper's 93-95% in OUR antipodal circle-crossing regime
(every path funnels through the centre). Re-reading the paper showed the gap is the
**scenario**, not the method: the paper's 93-95% is with *scattered* humans in a
*large* arena (standard 10x10 / 5 ppl ~0.995; challenging 15x15 / 10-20 ppl ~0.94),
robot crossing bottom->top. Architecture, reward (Eq 18-20), robot speed (1.0 m/s)
and the ORCA pedestrian model already match. **v27 trains in the paper's scattered
`paper_challenging` scenario** - a geometry-only probe (our current d_col 0.6 /
comfort 6) to measure how much the geometry alone closes the gap.

- **v18** remains the real-robot (0.26 m/s) baseline; **v22** the best result in the
  harder antipodal regime (84/74/66/38/36). Both kept for comparison.
- Architecture: SNCPPolicy = 3 NCP/LTC encoders + attention + actor-critic.
- Pedestrians: pure-Python **ORCA** (invisible robot - they avoid each other, not the robot).
- Best checkpoint = `min(success across holdout scenarios)`.

## Colab tips
- **Runtime -> Change runtime type -> A100** recommended (~3-5 h). T4/L4 work but slower.
- Mount Drive (Section 1.4) so checkpoints/logs survive a disconnect.


## 1. Setup

### 1.1 GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 1.2 Clone / update repository

Re-run after any push to pull the latest code. Note: this updates the repo files on the VM, **not** an already-open notebook — reopen the notebook from GitHub to get notebook changes.

In [ ]:
import os
REPO_URL = 'https://github.com/heimdilon/sncp-ppo-crowdnav.git'
REPO_DIR = '/content/sncp-ppo-crowdnav'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned. Pulling latest...')
    !cd {REPO_DIR} && git pull --rebase

%cd {REPO_DIR}
!git log --oneline -1

### 1.3 Install dependencies

In [ ]:
!pip install -q -r requirements.txt

import torch
print(f'torch     {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'          device: {torch.cuda.get_device_name(0)}')

import gymnasium, ncps, numpy, matplotlib
print(f'gymnasium {gymnasium.__version__}')
print(f'ncps      {ncps.__version__}')
print(f'numpy     {numpy.__version__}')

### 1.4 (Optional) Mount Google Drive

Set `USE_DRIVE = True` to persist `checkpoints/` and `logs/` across sessions (recommended for long runs — a disconnect mid-training otherwise loses everything).

In [ ]:
USE_DRIVE = False  # set True to persist runs across Colab sessions
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/sncp-ppo-crowdnav-runs'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(f'{DRIVE_PROJECT_DIR}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_PROJECT_DIR}/logs', exist_ok=True)
    import shutil
    for sub in ('checkpoints', 'logs'):
        local = f'{REPO_DIR}/{sub}'
        if os.path.islink(local):
            os.unlink(local)
        elif os.path.isdir(local):
            for f in os.listdir(local):
                dst = f'{DRIVE_PROJECT_DIR}/{sub}/{f}'
                if not os.path.exists(dst):
                    shutil.copy2(f'{local}/{f}', dst)
            shutil.rmtree(local)
        os.symlink(f'{DRIVE_PROJECT_DIR}/{sub}', local)
    print(f'Drive-backed dirs: {DRIVE_PROJECT_DIR}/{{checkpoints,logs}}')
else:
    print('Drive mount skipped (USE_DRIVE=False). Files are lost when the Colab session ends.')

## 2. Smoke tests

Fast sanity checks before spending GPU hours. Env + model first, then a 50-episode single-env training loop (legacy path) that exercises curriculum, holdout, value clipping, LR schedule, and the per-update diagnostics line.

In [ ]:
!python test_env.py

In [ ]:
!python test_model.py

In [ ]:
# 50-episode single-env smoke. NOT the full run (that's Section 3). Replay is 0
# here so this stays a quick baseline check of the single-env path.
import subprocess, sys
cmd = [
    sys.executable, '-u', '-m', 'sncp_ppo.train',
    '--episodes', '50',
    '--num_humans', '5',
    '--seed', '42',
    '--eval_freq', '25',
    '--holdout_episodes', '3',
    '--holdout_scenarios', 'easy', 'hard',
    '--update_freq', '5',
    '--log_freq', '10',
    '--curriculum_replay_ratio', '0.0',
    '--save_path', 'checkpoints/sncp_ppo_smoke.pt',
]
print('Running:', ' '.join(cmd))
print('=' * 80)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print(f'\nExited with code {p.returncode}')

## 3. Training (v32 - v30 + curriculum N->25 + budget 4M)

v27 trains from scratch in the paper's **`paper_challenging`** scenario (scattered
humans, 15x15 arena, robot 1.0 m/s, pedestrian parity). NO IL warm-start (v23's BC
warm-start regressed at high-N). A brief easy warmup (`--bootstrap_easy_steps`)
helps the cold fixed-N start get moving. This is the geometry-only probe; the
paper budget (challenging 50s / standard 12.5s), d_col 0.3 and comfort 2.0 are env-derived from the scenario.

| Argument | v27 value | note |
| --- | --- | --- |
| `--fixed_scenario` | `paper_challenging` | paper scattered / large-arena scenario |
| `--num_humans` | 10 | challenging density (sweep 10-20 at eval) |
| `--robot_vpref` | 1.0 | paper robot speed (peds parity via scenario) |
| `--lr` | 1e-4 | paper Table 1 |
| `--bootstrap_easy_steps` | 200000 | easy warmup before the fixed-N phase |
| `--holdout_scenarios` | paper_standard paper_challenging | paper-regime holdout |


### 3.1 Training run

In [ ]:
# v36 = v30 base + ALL levers combined (deliberately MULTI-variable): node-cap (v31) + reach/budget (v32)
# + multi-head (v33) + count-scaling (v29) + Beta with tuned entropy (v34 + --ent_coef) + sense-range (v35).
NUM_ENVS = 16
HORIZON = 128
TOTAL_STEPS = 4_000_000   # v32 budget (combined v36 run)
SEED = 42
LR = 1e-4
SAVE_PATH = 'checkpoints/sncp_ppo_v36.pt'

import subprocess, sys
cmd = [
    sys.executable, '-u', '-m', 'sncp_ppo.train',
    '--num_envs', str(NUM_ENVS),
    '--horizon', str(HORIZON),
    '--total_steps', str(TOTAL_STEPS),
    '--eval_freq_updates', '20',
    '--fixed_scenario', 'paper_challenging',
    '--num_humans', '10',
    '--num_humans_range', '10', '25',
    '--bootstrap_easy_steps', '200000',
    '--seed', str(SEED),
    '--lr', str(LR),
    '--lr_end_factor', '0.1',
    '--target_kl', '0.01',
    '--robot_vpref', '1.0',
    '--holdout_scenarios', 'paper_standard', 'paper_challenging',
    '--holdout_episodes', '50',
    '--pre_mlp',
    '--meanmax_pool',
    '--sense_range', '6.0',
    '--node_units', '256',
    '--node_output', '96',
    '--attn_heads', '4',
    '--attn_count_scaling',
    '--action_dist', 'beta',
    '--ent_coef', '0.001',
    '--save_path', SAVE_PATH,
]
print('Running:', ' '.join(cmd))
print('=' * 80)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end='')
p.wait()
print(f'\nExited with code {p.returncode}')
if p.returncode != 0:
    raise SystemExit(p.returncode)


### Resuming after a disconnect

There's no resume CLI. With `USE_DRIVE=True` the best checkpoint is safe in Drive; the simplest restart is to rerun 3.2 (optionally with a different `--seed`). The best-checkpoint logic keeps the highest-`min(success)` weights regardless of later collapse.

## 4. Evaluation (paper scenario)

Density sweep in the paper **`paper_challenging`** scenario at robot 1.0 m/s. Compare
DIRECTLY to the paper: challenging ~0.94 at 10-20 ppl (standard ~0.995 at 5). The
`run_post_eval` verdict compares to the antipodal `eval_v22` baseline - apples-to-
oranges here, so **expect verdict=fail**; read the density-sweep success rates
below directly. The cell is resilient: it surfaces the verdict and displays the
artifacts regardless of exit code (no more crash on a by-design fail verdict).


In [ ]:
CHECKPOINT = 'checkpoints/sncp_ppo_v36.pt'  # also used by the visualizers below
EVAL_OUT = 'eval_v36'
EVAL_SEED = 100
EVAL_EPISODES = 50

import subprocess, sys, os
cmd = [
    sys.executable, 'scripts/run_post_eval.py',
    '--version', '36',
    '--densities', '5', '10', '15', '20',
    '--scenario', 'paper_challenging',
    '--n_episodes', str(EVAL_EPISODES),
    '--seed', str(EVAL_SEED),
    '--trajectory_densities', '10', '20',
    '--robot_vpref', '1.0',
    '--human_vpref_override', '1.0',
    '--baseline_json', 'eval_v22/density_sweep.json',
    '--baseline_nav_steps', '32',
    '--nav_margin_steps', '8',
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd)
# run_post_eval exits non-zero when the comparison VERDICT is fail/warn. Here that
# is EXPECTED (paper scenario vs antipodal v22 baseline is not comparable) - it is
# NOT a crash. We do NOT use check=True; read the density sweep below directly.
if result.returncode != 0:
    print(f'\n[!] verdict = fail/warn (exit {result.returncode}) - expected vs the '
          'antipodal v22 baseline. Compare the density sweep to the paper: '
          'challenging ~0.94 at 10-20 ppl.')

from IPython.display import Image, Markdown, display
for name in ['report.md', 'training_diagnostics.md', 'artifact_verification.md']:
    path = f'{EVAL_OUT}/{name}'
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            display(Markdown(f.read()))
png = f'{EVAL_OUT}/density_sweep.png'
if os.path.exists(png):
    display(Image(png))


## 5. Visualize trajectories

Visualizers run in the v27 paper scenario (`paper_challenging`, robot 1.0 m/s) so
the plots reflect what the checkpoint actually learned. `CHECKPOINT` is set in
Section 4.


In [ ]:
# Single trajectory plot — first successful episode out of 20 tries.
!python scripts/visualize_trajectory.py \
    --checkpoint {CHECKPOINT} \
    --output trajectory_plot.png \
    --num_humans 10 \
    --scenario paper_challenging \
    --robot_vpref 1.0 --human_vpref_override 1.0 --human_goal_noise 0.0 --max_time 50 \
    --seed 42

from IPython.display import Image, display
display(Image('trajectory_plot.png'))

In [ ]:
# Animated GIF for a single scenario.
!python scripts/visualize_trajectory_gif.py --checkpoint {CHECKPOINT} --num_humans 10 --scenario paper_challenging

from IPython.display import Image, display
import glob
gifs = sorted(glob.glob('*.gif'))
if gifs:
    print(f'Generated: {gifs}')
    display(Image(gifs[-1]))

## 6. Training curves

Plots the newest training CSV and shows the v22 diagnostics + artifact-verification reports.

In [ ]:
import glob
csv_files = sorted(glob.glob('logs/training_*.csv'))
if not csv_files:
    print('No training CSVs found. Run Section 3 first.')
else:
    latest_csv = csv_files[-1]
    print(f'Plotting: {latest_csv}')
    !python scripts/plot_training.py --csv {latest_csv} --output training_curves_colab.png --window 50
    from IPython.display import Image, Markdown, display
    display(Image('training_curves_colab.png'))
    for path in ['eval_v36/training_diagnostics.md', 'eval_v36/artifact_verification.md']:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                display(Markdown(f.read()))
        except FileNotFoundError:
            print(f'{path} not found. Run the Section 4 evaluation cell first.')

### Inspect CSV in pandas (optional)

In [ ]:
import pandas as pd, glob
csv_files = sorted(glob.glob('logs/training_*.csv'))
if csv_files:
    df = pd.read_csv(csv_files[-1])
    print(f'Rows: {len(df)}')
    print(f'Columns: {list(df.columns)}')
    print('\nPhase distribution:')
    print(df['scenario'].value_counts().sort_index())
    print('\nHoldout success per eval (event points):')
    holdout_cols = [c for c in df.columns if c.startswith('holdout_') and c.endswith('_success')]
    if holdout_cols:
        hdf = df[holdout_cols].drop_duplicates()
        hdf.index = df.loc[hdf.index, 'episode']
        print(hdf.tail(10))

## 7. Persist results

With `USE_DRIVE=True` the checkpoint + logs are already in Drive. Otherwise set `DOWNLOAD = True` to grab the checkpoint, latest CSV + curve, and the full `eval_v35` evidence bundle before the session ends.

In [ ]:
from google.colab import files
import glob, os, shutil

DOWNLOAD = False  # set True to trigger browser download dialogs
if DOWNLOAD:
    if os.path.exists(SAVE_PATH):
        files.download(SAVE_PATH)
    for pattern in ['logs/training_*.csv', 'training_curves_colab.png']:
        for f in sorted(glob.glob(pattern))[-1:]:
            files.download(f)
    if os.path.isdir('eval_v36'):
        archive = shutil.make_archive('eval_v36_artifacts', 'zip', 'eval_v36')
        files.download(archive)

## 8. Notes & roadmap (current: v32 - v30 + curriculum N->25 + budget 4M)

`AGENTS.md` + the code are the source of truth.

### Story so far
- **v18** (paper-faithful `r_g`): real-robot (0.26 m/s) baseline, best generalist min 70%.
- **v22** (paper regime + LR 1e-4): best result in the harder ANTIPODAL regime
  (84/74/66/38/36), still ~37% at high N.
- **v23** (IL warm-start): regressed at N=10 (36->22); hypothesis disproven.
- **v26** (paper budget/geometry/comfort): re-reading the paper showed the gap is the SCENARIO, not the
  method - the paper's 93-95% is scattered humans / large arena, while ours was
  antipodal circle-crossing. v26 trains in the paper's `paper_challenging` scenario.

### Reading v27
- Compare the eval density sweep to the paper: challenging ~0.94 (10-20 ppl). The
  run_post_eval verdict (vs the antipodal v22 baseline) will say fail - ignore it,
  read the success rates.
- v27 = v26 + the paper Eq 11 pre-MLP (`--pre_mlp`); budget/geometry/comfort are
  env-derived from the scenario. Compare v27 local multi-seed sweep to v26 (74.8/61.6/53.2/43.6).
- Keep v22 as the harder-regime result for an honest write-up.
